**Exercise 1**

Given the following data:

| Tid | Refund | Marital Status | Taxable Income (K) | Cheat |
|-----|--------|----------------|--------------------|-------|
| 1   | Yes    | Single         | 125                | No    |
| 2   | No     | Married        | 100                | No    |
| 3   | No     | Single         | 70                 | No    |
| 4   | Yes    | Married        | 120                | No    |
| 5   | No     | Divorced       | 95                 | Yes   |
| 6   | No     | Married        | 60                 | No    |
| 7   | Yes    | Divorced       | 220                | No    |
| 8   | No     | Single         | 85                 | Yes   |
| 9   | No     | Married        | 75                 | No    |
| 10  | No     | Single         | 90                 | Yes   |


Between **Refund** and **Marital Status**, what is the best first split, using Gini?

If you have time, evaluate Gini for **Taxable Income (K)**.
*Note, for this feature, check quartile boundaries.*

$$Gini(t) = 1 - \sum_{i=1}^{c} (p_i)^2$$

In [1]:
# refund == 'Yes'
# 0 cheaters out of 3
round(1 - ((0/3)**2 + (3/3)**2), 2)

0.0

In [2]:
# refund == 'No'
# 3 cheaters out of 7
round(1 - ((3/7)**2 + (4/7)**2), 2)

0.49

In [3]:
# weighted average for Refund
round(0 * 3/10 + .49 * 7/10, 2)

0.34

In [4]:
# marital status == 'Single'
# 2 cheaters out of 4
round(1 - ((2/4)**2 + (2/4)**2), 2)

0.5

In [5]:
# marital status == 'Married'
# 0 cheaters out of 4
round(1 - ((0/4)**2 + (4/4)**2), 2)

0.0

In [6]:
# marital status == 'Divorced'
# 1 cheaters out of 2
round(1 - ((1/2)**2 + (1/2)**2), 2)

0.5

In [7]:
# weighted average for Marital Status
round(.5 * 4/10 + 0 * 4/10 + .5 * 2/10, 2)

0.3

In [ ]:
# taxable income
import pandas as pd

# Create DataFrame from sample data
data = {'Refund': ['Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'No'],
        'MaritalStatus': ['Single', 'Married', 'Single', 'Married', 'Divorced', 'Married', 'Divorced', 'Single', 'Married', 'Single'],
        'TaxableIncome': [125, 100, 70, 120, 95, 60, 220, 85, 75, 90],
        'Cheat': ['No', 'No', 'No', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes']}

df = pd.DataFrame(data)

# Calculate the quartiles of the "Taxable Income"
quartiles = df['TaxableIncome'].quantile([0.25, 0.5, 0.75])
quartiles

In [ ]:
def gini_impurity(subset):
    """ Calculate Gini impurity for a given subset of data. """
    if len(subset) == 0:
        return 0
    p = sum(subset['Cheat'] == 'Yes') / len(subset)
    return round(1 - p**2 - (1-p)**2, 2)

# Initialize list to store Gini impurities for each quartile split
split_results = []

# Calculate Gini for each quartile boundary
for q in quartiles:
    less_equal = df[df['TaxableIncome'] <= q]
    greater = df[df['TaxableIncome'] > q]
    gini_less_equal = gini_impurity(less_equal)
    gini_greater = gini_impurity(greater)

    # Weighted average Gini impurity
    total_records = len(df)
    weighted_gini = round((len(less_equal) / total_records) * gini_less_equal + (len(greater) / total_records) * gini_greater, 2)

    split_results.append({
        'Quartile Boundary': q,
        'Gini (<=)': gini_less_equal,
        'Gini (>)': gini_greater,
        'Weighted Gini': weighted_gini
    })

# Convert results to DataFrame for easier viewing
split_results_df = pd.DataFrame(split_results)
split_results_df

**Exercise 2**

Build a decision tree to fit the [federalist papers](https://www.kaggle.com/datasets/tobyanderson/federalist-papers) data. Note that you should restrict your analysis to papers written solely by Hamilton or Madison. Run your trained classifier on the "disputed" papers. What does your model tell you?

In [ ]:
federalist_papers = pd.read_csv('https://ist707.s3.us-east-2.amazonaws.com/data/federalist-papers.csv')
federalist_papers['author'].value_counts()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

train_df = federalist_papers[federalist_papers['author'].isin(['Hamilton', 'Madison'])]
X = train_df.iloc[:, 2:]
y = train_df['author']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

clf = DecisionTreeClassifier()
clf = clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 2))

disputed = federalist_papers[federalist_papers['author'] == 'dispt']
clf.predict(disputed.iloc[:, 2:])

In [ ]:
# Get feature importances
feature_importances = clf.feature_importances_

# Display feature importances in a DataFrame
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df)

**Exercise 3**

Build a **voting classifier** for the federalist papers, using all of the **non-ensemble** methods you've been exposed to in this class thus far (i.e., KNN, SVM, logistic regression, SGDClassifier, decision tree).

1) Compare this to a `RandomForest` classifier. Which works the best?

2) Compare this to a `GradientBoosting` classifier. Which works the best?

3) Add the `RandomForest` and `GradientBoosting` classifiers to your voting classifier. Does you performance improve?

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf = LogisticRegression(random_state=42)
svm_clf = SVC(probability=True, random_state=42) # 'probability=True' to enable soft voting
tree_clf = DecisionTreeClassifier(random_state=42)
knn_clf = KNN()
sgd_clf = SGDClassifier(random_state=42)

voting_hard_clf = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('svm', svm_clf),
        ('tree', tree_clf),
        ('knn', knn_clf),
        ('sgd', sgd_clf)
        ],
    voting='hard')
voting_hard_clf.fit(X_train, y_train)

# Evaluating classifiers
for clf in (log_clf, svm_clf, tree_clf, knn_clf, sgd_clf, voting_hard_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, round(accuracy_score(y_test, y_pred), 2))

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

rf_clf = RandomForestClassifier(random_state=42)
gb_clf = GradientBoostingClassifier(random_state=42)

voting_hard_clf = VotingClassifier(
    estimators=[
        ('lr', log_clf),
        ('svm', svm_clf),
        ('tree', tree_clf),
        ('knn', knn_clf),
        ('sgd', sgd_clf),
        ('rf', rf_clf),
        ('gb', gb_clf),
        ],
    voting='hard')
voting_hard_clf.fit(X_train, y_train)

# Evaluating classifiers
for clf in (log_clf, svm_clf, tree_clf, knn_clf, sgd_clf, rf_clf, gb_clf, voting_hard_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, round(accuracy_score(y_test, y_pred), 2))

**Exercise 4**

The "wine" dataset contains data about the chemical makeup of different varieties of wine and critics scores.  Use XGBoost to build a classifier for this data.  Manually tune the hyperparameters of the XGBoost model to try to achieve better accuracy on the test set than the baseline model. Some hyperparameters to consider tweaking:
   - `learning_rate`
   - `max_depth`
   - `n_estimators`
   - `gamma`
   - `subsample`
   - `colsample_bytree`

See [the online docs](https://xgboost.readthedocs.io/en/stable/parameter.html) for more info.

After tuning, use the `plot_importance` function again to see if feature importances have changed after tuning.


1. How did hyperparameter tuning affect the model's accuracy? Which hyperparameters seemed to have the most influence?
2. Did feature importances change after tuning? If so, why might that be?

In [ ]:
# Run this if you don't have XGBoost installed
%pip install XGBoost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import plot_importance

data = load_wine()

# We'll use a data frame to make sure we get real feature names out
X = pd.DataFrame(data.data,columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = xgb.XGBClassifier(objective='multi:softprob',
                        max_depth=4,
                        learning_rate=0.1,
                        n_estimators=100,
                        random_state=42)
clf.fit(X_train, y_train)

baseline_accuracy = clf.score(X_test, y_test)
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")

plot_importance(clf)
plt.show()